# MobilityAPI — Tutorial (streaming)

This notebook walks through the [OGC API – Moving Features – Part 4](https://www.opengis.net/spec/ogcapi-movingfeatures-4/1.0)
(Stream Extension) endpoints exposed by **MobilityAPI**, the thin compiled (Go)
tier over **MobilityDB/MEOS**, using the same one day of AIS (Automatic
Identification System) data from the [Danish Maritime Authority](http://aisdata.ais.dk/).

Each moving-feature quantity is delivered as Server-Sent Events from a continuous
query; its request–response counterpart,
[`tutorial/tutorial.ipynb`](../tutorial/tutorial.ipynb), retrieves the same
quantities with a `GET`.

## Relationship to the request–response tutorial

Both tutorials operate on the same `ships` collection and expose the same
moving-feature quantities — a feature, its speed-over-ground property, derived
measures, and a windowed aggregate. They differ only in interaction model: the
[request–response tutorial](../tutorial/tutorial.ipynb) retrieves each value with a
`GET`, whereas this notebook registers a continuous query and receives the values
over Server-Sent Events. The corresponding operations are:

| Quantity | Request–response | Streaming |
|---|---|---|
| A moving feature | `GET …/items/{id}` | the same vessel |
| Speed over ground | `GET …/tproperties/speed` | `POST …/queries` → SSE |
| Derived speed | `GET …/velocity` | a `transform` continuous query |
| Windowed aggregate | aggregate the returned series | a windowed continuous query |
| Visualization | a static plot | the live animated map |

## Setup

- The shared **`ships`** collection loaded once with the SAME script as the static
  tutorial (`tutorial/setup/load_ships.sql`).
- The Go tier built **with the streaming engine** (`-tags meos`) and reachable on
  `http://localhost:8088`; point elsewhere with `MFAPI_HOST`.
- A Jupyter kernel (the next cell installs `requests`).

In [ ]:
%pip install -q requests


In [ ]:
import os, json, requests
HOST = os.environ.get('MFAPI_HOST', 'http://localhost:8088')
S = requests.Session()
def show(r):
    print(r.status_code, r.request.method, r.url.replace(HOST,''))
    try: print(json.dumps(r.json(), indent=2)[:1200])
    except Exception: print(r.text[:600])
    return r
print('tier:', HOST, '->', S.get(HOST + '/health').json())


## A moving feature and its speed

The streaming tier serves the same `ships` collection as the request–response
tutorial: one day of Danish AIS, each vessel a `tgeompoint` trajectory with a
stored speed-over-ground `tfloat` (the AIS `SOG`, in knots). This section reads one
vessel's `speed` property; the sections below stream it.

In [ ]:
CID, PROP = 'ships', 'speed'
FID = int(os.environ.get('MFAPI_FID', '1'))   # most-travelled vessel by default

show(S.get(f'{HOST}/collections/{CID}'))
# the vessel's stored speed over ground (AIS SOG) — a tfloat in knots
show(S.get(f'{HOST}/collections/{CID}/items/{FID}/tproperties/{PROP}'))

## Continuous transform of a property

`POST …/tproperties/{name}/queries` registers a continuous query. The stored speed
is in knots; this query converts it to km/h (`× 1.852`) — a lifted scalar
multiplication applied to each streamed instant. The response is the OGC `cquery`
link object: a `queryId`, a `status`, and the `href` of the Server-Sent Events
stream.

Supported operations are the unary `ln, exp, log10, ceil, floor, abs, degrees,
radians, sin, cos, tan` and the scalar-argument `add, sub, mul, div`.

In [ ]:
q = S.post(f'{HOST}/collections/{CID}/items/{FID}/tproperties/{PROP}/queries',
           json={'operation': 'mul', 'arg': 1.852, 'intervalMs': 300})
show(q)
QID = q.json()['queryId']
STREAM = q.json()['href']

### Consuming the stream

The `href` is a Server-Sent Events endpoint. Each `instant` event carries the
transformed value at its timestamp — speed in km/h — as the vessel's track is
replayed. The next cell reads a few events and stops.

In [ ]:
import json as _json
def read_sse(url, n=10, timeout=30):
    with S.get(url, stream=True, timeout=timeout) as r:
        got = 0
        for line in r.iter_lines(decode_unicode=True):
            if line and line.startswith('data:'):
                ev = _json.loads(line[5:].strip())
                print(f"{ev['datetime']}  {ev['operation']}({ev['property']}) = {ev['value']:.3f} km/h")
                got += 1
                if got >= n: break
read_sse(STREAM, n=10)

### Trigonometric transforms

MEOS also lifts `sin`, `cos`, and `tan` over every streamed instant.
These are most useful on angular properties — compass headings, phase
angles, or any cyclic sensor datum. For illustration the next cell applies
`sin` to the speed-over-ground property; on an angular property the output
would be a unit-circle component of the bearing.

In [ ]:
q_sin = S.post(f'{HOST}/collections/{CID}/items/{FID}/tproperties/{PROP}/queries',
               json={'operation': 'sin', 'intervalMs': 300})
show(q_sin)
SIN_ID = q_sin.json()['queryId']

def read_trig(url, n=5, timeout=30):
    with S.get(url, stream=True, timeout=timeout) as r:
        got = 0
        for line in r.iter_lines(decode_unicode=True):
            if line and line.startswith('data:'):
                ev = _json.loads(line[5:].strip())
                print(f"{ev['datetime']}  {ev['operation']}({ev['property']}) = {ev['value']:.6f}")
                got += 1
                if got >= n: break
read_trig(q_sin.json()['href'], n=5)
S.delete(f'{HOST}/collections/{CID}/items/{FID}/tproperties/{PROP}/queries/{SIN_ID}')

## Windowed aggregation

A continuous query can also aggregate over a window (OGC MF – Part 4): `AVG`, `SUM`,
`MIN`, `MAX` or `COUNT` over a `COUNT`, `TUMBLING` or `HOPPING` window. Each result
carries the aggregate value together with its window bounds.

In [ ]:
agg = S.post(f'{HOST}/collections/{CID}/items/{FID}/tproperties/{PROP}/queries',
             json={'aggregation': 'AVG', 'window': {'type': 'COUNT', 'size': 3}, 'intervalMs': 300})
show(agg)
AID = agg.json()['queryId']
def read_agg(url, n=3, timeout=30):
    with S.get(url, stream=True, timeout=timeout) as r:
        got = 0
        for line in r.iter_lines(decode_unicode=True):
            if line and line.startswith('data:'):
                ev = _json.loads(line[5:].strip())
                print(f"[{ev['windowStart']} .. {ev['windowEnd']}]  {ev['aggregation']}({ev['property']}) = {ev['value']:.3f}  (n={ev['count']})")
                got += 1
                if got >= n: break
read_agg(agg.json()['href'], n=3)
S.delete(f'{HOST}/collections/{CID}/items/{FID}/tproperties/{PROP}/queries/{AID}')


## Visualization — the animated fleet

The sections above retrieve streamed values through the API. The map below animates
the whole fleet of the same `ships` data on a single temporal clock. Its
distinctive property is that MEOS executes in the browser (WebAssembly): the tier
transmits each vessel's trajectory once — MEOS-sampled to a visualization interval
and decoupled into a `path` and parallel `timestamps` — and MEOS.js reconstructs a
`TGeomPoint` per vessel and computes every vessel's position with `valueAtTimestamp`
on each animation frame. DeckGL renders the resulting positions over a MapLibre
basemap, with a play/pause, scrubber and pace controller.

![Animated fleet — MEOS.js computing every ship's position live, in the browser](map/fleet.gif)

▶ **[See the live, interactive animation](http://localhost:5174/fleet.html)** — run
`npm run build && npm run preview` in `tutorial-stream/map`, then open the URL to
zoom, pan, scrub the timeline and change the pace over the full fleet.

In [ ]:
# the animated fleet map — run `npm run build && npm run preview` in tutorial-stream/map first.
# When this notebook is executed live, the cell renders the interactive map inline;
# GitHub's static view shows the GIF above instead.
from IPython.display import IFrame
MAP = os.environ.get('MFAPI_MAP', 'http://localhost:5174/fleet.html')
IFrame(f'{MAP}?cid={CID}', width='100%', height=560)

## Summary

- A stored `tfloat` (the vessel's AIS speed over ground) is transformed by a lifted
  MEOS function in process, per record; the transform is exact because each streamed
  record is an instant.
- Trigonometric transforms (`sin`, `cos`, `tan`) lift pointwise over every streamed
  instant, enabling angular-property processing (headings, phase data) with no
  client-side code.
- A windowed aggregation reduces the stream over `COUNT`, `TUMBLING` or `HOPPING`
  windows.
- The request–response tutorial reads the same property and derived measures with
  `GET`; this notebook receives them over Server-Sent Events. The data and the MEOS
  computation are identical — only the delivery differs.

## Query lifecycle

The streaming API exposes a query lifecycle: `GET …/queries/{queryId}` reports the
status and `DELETE` stops the query (`registered → running → stopped`), identically
across engines.

In [ ]:
show(S.get(f'{HOST}/collections/{CID}/items/{FID}/tproperties/{PROP}/queries/{QID}'))
show(S.delete(f'{HOST}/collections/{CID}/items/{FID}/tproperties/{PROP}/queries/{QID}'))
# after stopping, the query is gone
print('after stop:', S.get(f'{HOST}/collections/{CID}/items/{FID}/tproperties/{PROP}/queries/{QID}').status_code)


### Engines

The cells above run on the default in-process `meos-local` engine. Because the
control plane is engine-neutral, the same notebook runs unchanged on a cluster
engine; the engine is selected where the tier starts, not in the client:

```
# in-process MEOS (default)
MFAPI_DSN=<dsn> ./mfapi

# Flink: run each continuous query as a Flink DataStream job (see flink/README.md)
MFAPI_STREAM_ENGINE=flink \
MFAPI_FLINK_CMD="java <opens> -cp <classpath> MfStreamBridgeJob" \
MFAPI_FLINK_LIBPATH=<libmeos-dir> \
MFAPI_DSN=<dsn> ./mfapi
```

The `cquery` link object, the SSE stream, and the lifecycle are identical across
engines — a Kafka Streams or Spark Structured Streaming engine plugs into the same
seam through `MFAPI_FLINK_CMD`. This mirrors the request–response tutorial, where
`MFAPI_DSN`'s scheme (`postgres://`, `duckdb:`, `spark:`) switches the database
backend with no change to the notebook.